## Label Scenario 2 



In [2]:
#
# Cell 1: Libraries laden
#

import pandas as pd
import numpy as np
import wfdb
import ast

In [6]:
#
# Cell 2:Path for input data and definition of sampling rate
#

#path = 'C:/Users/Buchh/OneDrive/Leuphana_Data_Science/Masterarbeit/ECG/Python/Daten_einlesen/test_data/'
path = 'E:/MA_Daten_2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'
sampling_rate=100


In [8]:
#
# Cell 3: Definition of functions
#

def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

def aggregate_form_rhythm(y_dic): #BuK
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].form_rhythm)
    return list(set(tmp))


In [10]:
#
# Cell 3a: Specify the directory name for loading of data
#

from pathlib import Path

directory_path = Path("D:/ECG/X_zero")

In [12]:
#
# Cell 4:Loading stored raw data, if available. 
#

#X = np.load('X_np_array.npy')
Y = pd.read_pickle(directory_path/'Y_data.pkl')

In [14]:
#
# Cell 5: FORM AND RHYTHM LABELS
# Load scp_statements.csv for form and rhythm aggregation (BuK)
#

agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[(agg_df.form == 1) | (agg_df.rhythm == 1)]
agg_df['form_rhythm'] = agg_df.index

In [16]:
#
#Cell 6: Apply form and rhythm class, BuK
#

Y['label'] = Y.scp_codes.apply(aggregate_form_rhythm)
display(Y)

,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,label
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr,"[SR, LVOLT]"
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr,[SBRAD]
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr,[SR]
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr,[SR]
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr,[SR]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21794,21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,...,NaN,", alles,",NaN,NaN,1ES,NaN,7,records100/21000/21833_lr,records500/21000/21833_hr,"[PVC, VCLVH, STACH, NDT]"
21795,21834,20703.0,300.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-06-05 11:33:39,...,NaN,NaN,NaN,NaN,NaN,NaN,4,records100/21000/21834_lr,records500/21000/21834_hr,"[ABQRS, SR]"
21796,21835,19311.0,59.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-08 10:30:27,...,NaN,", I-AVR,",NaN,NaN,NaN,NaN,2,records100/21000/21835_lr,records500/21000/21835_hr,[SR]
21797,21836,8873.0,64.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-09 18:21:49,...,NaN,NaN,NaN,NaN,SVES,NaN,8,records100/21000/21836_lr,records500/21000/21836_hr,[SR]


In [18]:
#
#Cell 7: Make a copy from Y for further manipulation
#

Y_copy = Y[['ecg_id','patient_id','strat_fold','label']].copy() 
Y_copy

,ecg_id,patient_id,strat_fold,label
0,1,15709.0,3,"[SR, LVOLT]"
1,2,13243.0,2,[SBRAD]
2,3,20372.0,5,[SR]
3,4,17014.0,3,[SR]
4,5,17448.0,4,[SR]
...,...,...,...,...
21794,21833,17180.0,7,"[PVC, VCLVH, STACH, NDT]"
21795,21834,20703.0,4,"[ABQRS, SR]"
21796,21835,19311.0,2,[SR]
21797,21836,8873.0,8,[SR]


## Y-data manipulation for augmentation starts 

In [18]:
# Cell 7a: 

from pathlib import Path
directory_path_augm = Path("d:/ECG/X_augmented")

In [20]:
# Cell 7b: loads dicitionary with augmented X-xata

import pickle

with open(directory_path_augm/'X_augmented_accomp.pkl', 'rb') as f:
    X_augmented_accomp = pickle.load(f) # deserialize using load()
f.close()  

X_augmented_accomp

{'# of additional X-Data concatenated': 2}

In [22]:
# Cell 7c: select a certain row for later control purposes:
Y_21794 = Y_copy[Y_copy.index==21794].copy()
Y_21794

,ecg_id,patient_id,strat_fold,label
21794,21833,17180.0,7,"[VCLVH, NDT, PVC, STACH]"


In [24]:
# Cell 7d: concatenates the df Y_copy according to the value of  
# '# of additional X-Data concatenated':
Y_add=Y_copy
for i in range(0, X_augmented_accomp.get('# of additional X-Data concatenated')):
    Y_copy= pd.concat([Y_copy,Y_add], axis=0, ignore_index= True).copy()

Y_copy    

,ecg_id,patient_id,strat_fold,label
0,1,15709.0,3,"[SR, LVOLT]"
1,2,13243.0,2,[SBRAD]
2,3,20372.0,5,[SR]
3,4,17014.0,3,[SR]
4,5,17448.0,4,[SR]
...,...,...,...,...
65392,21833,17180.0,7,"[VCLVH, NDT, PVC, STACH]"
65393,21834,20703.0,4,"[SR, ABQRS]"
65394,21835,19311.0,2,[SR]
65395,21836,8873.0,8,[SR]


In [26]:
# Cell 7e: compare two df extracts for control of correct index assignment  
# 21749 + 21799 = 43593

Y_43593 = Y_copy[Y_copy.index==43593].copy()
display(Y_43593)
display(Y_21794)

,ecg_id,patient_id,strat_fold,label
43593,21833,17180.0,7,"[VCLVH, NDT, PVC, STACH]"


,ecg_id,patient_id,strat_fold,label
21794,21833,17180.0,7,"[VCLVH, NDT, PVC, STACH]"


In [28]:
Y_65392 = Y_copy[Y_copy.index==65392].copy()
display(Y_65392)

,ecg_id,patient_id,strat_fold,label
65392,21833,17180.0,7,"[VCLVH, NDT, PVC, STACH]"


## Y-data manipulation for augmentation ends
## The remaining cells are indentical with the cells of the modul for not augmented data   

In [20]:
#
# Cell 8: in all combinations of labels with "SR" remove "SR" 
# clean list of labels is now stored in the column "codes"
#

Y_copy["codes"] = Y_copy["label"].apply(
    lambda lst: ["SR"] if len(lst) == 1 and lst[0] == "SR"
    else sorted([c for c in lst if c != "SR"])
)
Y_copy

,ecg_id,patient_id,strat_fold,label,codes
0,1,15709.0,3,"[SR, LVOLT]",[LVOLT]
1,2,13243.0,2,[SBRAD],[SBRAD]
2,3,20372.0,5,[SR],[SR]
3,4,17014.0,3,[SR],[SR]
4,5,17448.0,4,[SR],[SR]
...,...,...,...,...,...
21794,21833,17180.0,7,"[PVC, VCLVH, STACH, NDT]","[NDT, PVC, STACH, VCLVH]"
21795,21834,20703.0,4,"[ABQRS, SR]",[ABQRS]
21796,21835,19311.0,2,[SR],[SR]
21797,21836,8873.0,8,[SR],[SR]


In [22]:
#
# Cell 9: count the items in the lists in column "codes" and write to "len" column 
#

Y_copy["len"] = Y_copy["codes"].apply(
    lambda lst: len(lst)
)
Y_copy


,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[SR, LVOLT]",[LVOLT],1
1,2,13243.0,2,[SBRAD],[SBRAD],1
2,3,20372.0,5,[SR],[SR],1
3,4,17014.0,3,[SR],[SR],1
4,5,17448.0,4,[SR],[SR],1
...,...,...,...,...,...,...
21794,21833,17180.0,7,"[PVC, VCLVH, STACH, NDT]","[NDT, PVC, STACH, VCLVH]",4
21795,21834,20703.0,4,"[ABQRS, SR]",[ABQRS],1
21796,21835,19311.0,2,[SR],[SR],1
21797,21836,8873.0,8,[SR],[SR],1


In [24]:
#
# Cell 10: selects ECGs with non empty code only
#

Y_copy = Y_copy[Y_copy["len"]>0].copy()
Y_copy

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[SR, LVOLT]",[LVOLT],1
1,2,13243.0,2,[SBRAD],[SBRAD],1
2,3,20372.0,5,[SR],[SR],1
3,4,17014.0,3,[SR],[SR],1
4,5,17448.0,4,[SR],[SR],1
...,...,...,...,...,...,...
21794,21833,17180.0,7,"[PVC, VCLVH, STACH, NDT]","[NDT, PVC, STACH, VCLVH]",4
21795,21834,20703.0,4,"[ABQRS, SR]",[ABQRS],1
21796,21835,19311.0,2,[SR],[SR],1
21797,21836,8873.0,8,[SR],[SR],1


Cell 10 to cell 17: all ECGs with one label
Cell ... to cel ...: all ECGs with two label

In [26]:
#
# Cell 11: filter all instaces with one label
#

Y_label_len_one_df = Y_copy[Y_copy["len"]==1].copy()
Y_label_len_one_df

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[SR, LVOLT]",[LVOLT],1
1,2,13243.0,2,[SBRAD],[SBRAD],1
2,3,20372.0,5,[SR],[SR],1
3,4,17014.0,3,[SR],[SR],1
4,5,17448.0,4,[SR],[SR],1
...,...,...,...,...,...,...
21793,21832,7954.0,7,[SR],[SR],1
21795,21834,20703.0,4,"[ABQRS, SR]",[ABQRS],1
21796,21835,19311.0,2,[SR],[SR],1
21797,21836,8873.0,8,[SR],[SR],1


In [28]:
#
# Cell 12:ONE LABEL - convert list element of column "codes" into objects
#

Y_label_len_one_df['codes'] = Y_label_len_one_df['codes'].apply(lambda x: "".join(x)).copy()
Y_label_len_one_df

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[SR, LVOLT]",LVOLT,1
1,2,13243.0,2,[SBRAD],SBRAD,1
2,3,20372.0,5,[SR],SR,1
3,4,17014.0,3,[SR],SR,1
4,5,17448.0,4,[SR],SR,1
...,...,...,...,...,...,...
21793,21832,7954.0,7,[SR],SR,1
21795,21834,20703.0,4,"[ABQRS, SR]",ABQRS,1
21796,21835,19311.0,2,[SR],SR,1
21797,21836,8873.0,8,[SR],SR,1


In [30]:
#
# Cell 12a: only information
#

Y_label_len_one_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17504 entries, 0 to 21798
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ecg_id      17504 non-null  int64  
 1   patient_id  17504 non-null  float64
 2   strat_fold  17504 non-null  int64  
 3   label       17504 non-null  object 
 4   codes       17504 non-null  object 
 5   len         17504 non-null  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 957.2+ KB


In [32]:
#
# Cell 13: ONE LABEL - count number of each code in column "codes" 
#

combo_counts = Y_label_len_one_df["codes"].value_counts()
combo_counts

#combo_counts.to_csv("combo_counts.csv")

codes
SR        10125
ABQRS      2332
NDT         862
AFIB        481
SARRH       472
SBRAD       428
VCLVH       399
STACH       370
PVC         316
PACE        281
NST_        271
QWAVE       224
STD_        142
LPR         136
PAC         127
NT_         101
LVOLT        88
SVARR        74
LNGQT        62
INVT         52
HVOLT        43
AFLT         28
LOWT         27
DIG          13
TAB_         12
STE_         11
BIGU         10
PSVT          8
SVTAC         5
PRC(S)        3
TRIGU         1
Name: count, dtype: int64

In [34]:
#
# Cell 13a: information 
#

type(combo_counts)

pandas.core.series.Series

In [36]:
#
# Cell 14: ONE LABEL - coverts series into df 
#

combo_counts_df = combo_counts.to_frame()
combo_counts_df

,count
codes,
SR,10125
ABQRS,2332
NDT,862
AFIB,481
SARRH,472
SBRAD,428
VCLVH,399
STACH,370
PVC,316


In [38]:
#
# Cell 14a: information
#

combo_counts_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31 entries, SR to TRIGU
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   count   31 non-null     int64
dtypes: int64(1)
memory usage: 1.5+ KB


## Important note: to be strinctly allign with the number of non augmented data instances the threshold value of data taken into account must be changed to 3 x 49 = 147. Otherwise the number of augmented data are not simply the threefold number of the non augmented data.      

In [40]:
#
# Cell 15: ONE LABEL - restrict codes to n.l.t. 49 instances  
#

combo_count_short_df=combo_counts_df[combo_counts_df['count']>=49]
combo_count_short_df

,count
codes,
SR,10125
ABQRS,2332
NDT,862
AFIB,481
SARRH,472
SBRAD,428
VCLVH,399
STACH,370
PVC,316


In [42]:
#
# Cell 15a: ONE LABEL - restrict codes to l.t. 49 instances  
#

combo_count_remaining_df=combo_counts_df[combo_counts_df['count']<49]
combo_count_remaining_df

,count
codes,
HVOLT,43
AFLT,28
LOWT,27
DIG,13
TAB_,12
STE_,11
BIGU,10
PSVT,8
SVTAC,5


In [44]:
#
# Cell 16: ONE LABEL - reorders df  
#

combo_count_short_reset_df = combo_count_short_df.reset_index( ) 
combo_count_short_reset_df

,codes,count
0,SR,10125
1,ABQRS,2332
2,NDT,862
3,AFIB,481
4,SARRH,472
5,SBRAD,428
6,VCLVH,399
7,STACH,370
8,PVC,316
9,PACE,281


In [46]:
#
# Cell 16a: ONE LABEL - reorders df  
#

combo_count_remaining_reset_df = combo_count_remaining_df.reset_index( ) 
combo_count_remaining_reset_df

,codes,count
0,HVOLT,43
1,AFLT,28
2,LOWT,27
3,DIG,13
4,TAB_,12
5,STE_,11
6,BIGU,10
7,PSVT,8
8,SVTAC,5
9,PRC(S),3


In [48]:
#
# Cell 16b: information  
#

combo_count_short_reset_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   codes   20 non-null     object
 1   count   20 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 452.0+ bytes


In [50]:
#
# Cell 17: ONE LABEL - convert column "codes" to list  
#

codes_list =combo_count_short_reset_df['codes'].to_list() 
codes_list
#df = df[df.index.isin(df1.index)]

['SR',
 'ABQRS',
 'NDT',
 'AFIB',
 'SARRH',
 'SBRAD',
 'VCLVH',
 'STACH',
 'PVC',
 'PACE',
 'NST_',
 'QWAVE',
 'STD_',
 'LPR',
 'PAC',
 'NT_',
 'LVOLT',
 'SVARR',
 'LNGQT',
 'INVT']

In [52]:
#
# Cell 18: ONE LABEL - convert column "codes" to list  
#

codes_list_remaining =combo_count_remaining_reset_df['codes'].to_list() 
codes_list_remaining
#df = df[df.index.isin(df1.index)]

['HVOLT',
 'AFLT',
 'LOWT',
 'DIG',
 'TAB_',
 'STE_',
 'BIGU',
 'PSVT',
 'SVTAC',
 'PRC(S)',
 'TRIGU']

In [54]:
#
# Cell 19: ONE LABEL - filters from df "Y_label_len_one_df" all rows with codes in column "codes" 
# that are contained in the list codeslistconvert column "codes" to list
#

Y_label_len_one_short_df= Y_label_len_one_df[Y_label_len_one_df['codes'].isin(codes_list)].copy()
Y_label_len_one_short_df


,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[SR, LVOLT]",LVOLT,1
1,2,13243.0,2,[SBRAD],SBRAD,1
2,3,20372.0,5,[SR],SR,1
3,4,17014.0,3,[SR],SR,1
4,5,17448.0,4,[SR],SR,1
...,...,...,...,...,...,...
21793,21832,7954.0,7,[SR],SR,1
21795,21834,20703.0,4,"[ABQRS, SR]",ABQRS,1
21796,21835,19311.0,2,[SR],SR,1
21797,21836,8873.0,8,[SR],SR,1


In [56]:
#
#Cell 20: ONE LABEL - filters from df "Y_label_len_one_df" all rows with codes in column "codes" 
# that are contained in the list codeslistconvert column "codes" to list
#

Y_label_len_one_remaining_df= Y_label_len_one_df[Y_label_len_one_df['codes'].isin(codes_list_remaining)].copy()
Y_label_len_one_remaining_df


,ecg_id,patient_id,strat_fold,label,codes,len
17,18,13619.0,9,[AFLT],AFLT,1
22,23,13619.0,9,[AFLT],AFLT,1
33,34,13619.0,9,[AFLT],AFLT,1
186,194,20434.0,8,[TAB_],TAB_,1
199,207,17633.0,8,"[HVOLT, SR]",HVOLT,1
...,...,...,...,...,...,...
18062,18100,3042.0,2,"[SR, LOWT]",LOWT,1
18076,18114,7871.0,4,"[HVOLT, SR]",HVOLT,1
18327,18366,3964.0,7,"[BIGU, SR]",BIGU,1
19604,19643,21602.0,2,"[DIG, SR]",DIG,1


In [58]:
#
# Cell 21: Separates the ECGs with SR label not wnated by now (but later).
#

Y_SR = Y_label_len_one_short_df[Y_label_len_one_short_df['codes'].str.contains(r'^(?=.*SR)')].copy()
Y_SR

,ecg_id,patient_id,strat_fold,label,codes,len
2,3,20372.0,5,[SR],SR,1
3,4,17014.0,3,[SR],SR,1
4,5,17448.0,4,[SR],SR,1
5,6,19005.0,4,[SR],SR,1
6,7,16193.0,7,[SR],SR,1
...,...,...,...,...,...,...
21792,21831,11905.0,9,[SR],SR,1
21793,21832,7954.0,7,[SR],SR,1
21796,21835,19311.0,2,[SR],SR,1
21797,21836,8873.0,8,[SR],SR,1


In [60]:
#
# Cell 22: 
#

Y_SR.loc[:, 'codes_2'] = 'SR'
Y_SR

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
2,3,20372.0,5,[SR],SR,1,SR
3,4,17014.0,3,[SR],SR,1,SR
4,5,17448.0,4,[SR],SR,1,SR
5,6,19005.0,4,[SR],SR,1,SR
6,7,16193.0,7,[SR],SR,1,SR
...,...,...,...,...,...,...,...
21792,21831,11905.0,9,[SR],SR,1,SR
21793,21832,7954.0,7,[SR],SR,1,SR
21796,21835,19311.0,2,[SR],SR,1,SR
21797,21836,8873.0,8,[SR],SR,1,SR


In [62]:
#
# Cell 23: Substract the not wanted SR ECGs from Y_copy (all ECGs)
#

Y_label_len_one_short_df = Y_label_len_one_short_df[~Y_label_len_one_short_df.index.isin(Y_SR.index)].copy()
Y_label_len_one_short_df

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[SR, LVOLT]",LVOLT,1
1,2,13243.0,2,[SBRAD],SBRAD,1
7,8,11275.0,9,"[ABQRS, SR]",ABQRS,1
10,11,11243.0,5,[SARRH],SARRH,1
11,12,11031.0,8,[SBRAD],SBRAD,1
...,...,...,...,...,...,...
21777,21816,15103.0,3,"[SR, NDT]",NDT,1
21778,21817,18354.0,9,[PACE],PACE,1
21785,21824,19977.0,8,"[ABQRS, SR]",ABQRS,1
21790,21829,14508.0,3,"[ABQRS, SR]",ABQRS,1


In [64]:
#
# Cell 24: 
#

Y_label_len_one_short_df.loc[:, 'codes_2'] = 'single_label'
Y_label_len_one_short_df

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
0,1,15709.0,3,"[SR, LVOLT]",LVOLT,1,single_label
1,2,13243.0,2,[SBRAD],SBRAD,1,single_label
7,8,11275.0,9,"[ABQRS, SR]",ABQRS,1,single_label
10,11,11243.0,5,[SARRH],SARRH,1,single_label
11,12,11031.0,8,[SBRAD],SBRAD,1,single_label
...,...,...,...,...,...,...,...
21777,21816,15103.0,3,"[SR, NDT]",NDT,1,single_label
21778,21817,18354.0,9,[PACE],PACE,1,single_label
21785,21824,19977.0,8,"[ABQRS, SR]",ABQRS,1,single_label
21790,21829,14508.0,3,"[ABQRS, SR]",ABQRS,1,single_label


In [66]:
#
# Cell 25: TWO LABEL -  filter all instances with two label
#

Y_label_len_two_df = Y_copy[Y_copy["len"]==2].copy()
Y_label_len_two_df

,ecg_id,patient_id,strat_fold,label,codes,len
19,20,13619.0,9,"[ABQRS, AFLT]","[ABQRS, AFLT]",2
25,26,13619.0,9,"[DIG, NST_, SR]","[DIG, NST_]",2
27,28,13619.0,9,"[DIG, NST_, SR]","[DIG, NST_]",2
102,103,13722.0,4,"[ABQRS, STACH]","[ABQRS, STACH]",2
162,170,15691.0,9,"[PVC, NST_, SR]","[NST_, PVC]",2
...,...,...,...,...,...,...
21773,21812,20789.0,10,"[ABQRS, SVARR]","[ABQRS, SVARR]",2
21779,21818,19204.0,10,"[ABQRS, VCLVH, SR]","[ABQRS, VCLVH]",2
21787,21826,9178.0,10,"[ABQRS, SARRH]","[ABQRS, SARRH]",2
21788,21827,13862.0,5,"[ABQRS, SVARR]","[ABQRS, SVARR]",2


In [68]:
#
# Cell 25a: for information only
#

x = Y_label_len_two_df["codes"].iloc[0]

print("type:", type(x))
print("value:", x)

if isinstance(x, list):
    print("list length:", len(x))
    print("element types (unique):", {type(e) for e in x})
    print("first 20 elements (repr):", [repr(e) for e in x[:20]])

type: <class 'list'>
value: ['ABQRS', 'AFLT']
list length: 2
element types (unique): {<class 'str'>}
first 20 elements (repr): ["'ABQRS'", "'AFLT'"]


In [70]:
#
# Cell 26: TWO LABEL - convert list element of column "codes" into objects
#

Y_label_len_two_df['codes'] = Y_label_len_two_df['codes'].apply(lambda x: ", ".join(x)).copy()
#Y_label_len_two_df['codes'] = Y_label_len_two_df['codes'].apply(lambda x: [str(j) for j in x]).copy()
Y_label_len_two_df


,ecg_id,patient_id,strat_fold,label,codes,len
19,20,13619.0,9,"[ABQRS, AFLT]","ABQRS, AFLT",2
25,26,13619.0,9,"[DIG, NST_, SR]","DIG, NST_",2
27,28,13619.0,9,"[DIG, NST_, SR]","DIG, NST_",2
102,103,13722.0,4,"[ABQRS, STACH]","ABQRS, STACH",2
162,170,15691.0,9,"[PVC, NST_, SR]","NST_, PVC",2
...,...,...,...,...,...,...
21773,21812,20789.0,10,"[ABQRS, SVARR]","ABQRS, SVARR",2
21779,21818,19204.0,10,"[ABQRS, VCLVH, SR]","ABQRS, VCLVH",2
21787,21826,9178.0,10,"[ABQRS, SARRH]","ABQRS, SARRH",2
21788,21827,13862.0,5,"[ABQRS, SVARR]","ABQRS, SVARR",2


In [72]:
#
# Cell 27: TWO LABEL - count number of each code in column "codes" 
#

combo_counts_two = Y_label_len_two_df["codes"].value_counts()
combo_counts_two

#combo_counts.to_csv("combo_counts.csv")

codes
ABQRS, AFIB     224
NDT, NT_        148
AFIB, NST_      142
ABQRS, PVC      120
ABQRS, STACH    119
               ... 
PACE, PVC         1
NST_, TRIGU       1
PACE, STE_        1
BIGU, NDT         1
AFIB, PSVT        1
Name: count, Length: 192, dtype: int64

In [74]:
#
# Cell 21: TWO LABEL - coverts series into df 
#

combo_counts_two_df = combo_counts_two.to_frame()
combo_counts_two_df

,count
codes,
"ABQRS, AFIB",224
"NDT, NT_",148
"AFIB, NST_",142
"ABQRS, PVC",120
"ABQRS, STACH",119
...,...
"PACE, PVC",1
"NST_, TRIGU",1
"PACE, STE_",1


In [76]:
#
# Cell 22: TWO LABEL - restrict codes to n.l.t. 49 instances  
#

combo_count_two_short_df=combo_counts_two_df[combo_counts_two_df['count']>=49]
combo_count_two_short_df

,count
codes,
"ABQRS, AFIB",224
"NDT, NT_",148
"AFIB, NST_",142
"ABQRS, PVC",120
"ABQRS, STACH",119
"AFIB, NDT",100
"ABQRS, SARRH",83
"ABQRS, SBRAD",74
"AFIB, PVC",63


In [78]:
#
# Cell 23: TWO LABEL - restrict codes to n.l.t. 49 instances  
#

combo_count_two_reamining_df=combo_counts_two_df[combo_counts_two_df['count']<49].copy()
combo_count_two_reamining_df

,count
codes,
"ABQRS, SVARR",44
"ABQRS, VCLVH",42
"NDT, STACH",36
"NDT, STD_",36
"NST_, STACH",34
...,...
"PACE, PVC",1
"NST_, TRIGU",1
"PACE, STE_",1


In [80]:
#
# Cell 24: TWO LABEL - reorders df  
#

combo_count_two_short_reset_df = combo_count_two_short_df.reset_index( ) 
combo_count_two_short_reset_df

,codes,count
0,"ABQRS, AFIB",224
1,"NDT, NT_",148
2,"AFIB, NST_",142
3,"ABQRS, PVC",120
4,"ABQRS, STACH",119
5,"AFIB, NDT",100
6,"ABQRS, SARRH",83
7,"ABQRS, SBRAD",74
8,"AFIB, PVC",63
9,"PVC, SARRH",59


In [82]:
#
# Cell 25: TWO LABEL - reorders df  
#

combo_count_two_reamining_reset_df = combo_count_two_reamining_df.reset_index( ) 
combo_count_two_reamining_reset_df

,codes,count
0,"ABQRS, SVARR",44
1,"ABQRS, VCLVH",42
2,"NDT, STACH",36
3,"NDT, STD_",36
4,"NST_, STACH",34
...,...,...
171,"PACE, PVC",1
172,"NST_, TRIGU",1
173,"PACE, STE_",1
174,"BIGU, NDT",1


In [84]:
#
# Cell 26: TWO LABEL - convert column "codes" to list  
#

codes_two_list =combo_count_two_short_reset_df['codes'].to_list() 
codes_two_list
#df = df[df.index.isin(df1.index)]


['ABQRS, AFIB',
 'NDT, NT_',
 'AFIB, NST_',
 'ABQRS, PVC',
 'ABQRS, STACH',
 'AFIB, NDT',
 'ABQRS, SARRH',
 'ABQRS, SBRAD',
 'AFIB, PVC',
 'PVC, SARRH',
 'INVT, STD_',
 'ABQRS, NDT',
 'STD_, VCLVH',
 'LOWT, STD_',
 'NDT, PVC',
 'LOWT, NDT']

In [86]:
#
# Cell 27: TWO LABEL - convert column "codes" to list  
#

codes_two_list_remaining =combo_count_two_reamining_reset_df['codes'].to_list() 
codes_two_list_remaining
#df = df[df.index.isin(df1.index)]


['ABQRS, SVARR',
 'ABQRS, VCLVH',
 'NDT, STACH',
 'NDT, STD_',
 'NST_, STACH',
 'PVC, STACH',
 'NST_, STD_',
 'NDT, SARRH',
 'AFIB, STD_',
 'AFIB, VCLVH',
 'NDT, VCLVH',
 'QWAVE, STD_',
 'BIGU, PVC',
 'PAC, STD_',
 'SBRAD, VCLVH',
 'STACH, STD_',
 'NST_, VCLVH',
 'NDT, SBRAD',
 'AFIB, QWAVE',
 'PAC, PVC',
 'LVOLT, NDT',
 'BIGU, PAC',
 'LPR, STD_',
 'NT_, VCLVH',
 'NT_, QWAVE',
 'AFIB, DIG',
 'NDT, QWAVE',
 'NT_, STACH',
 'INVT, QWAVE',
 'NT_, PAC',
 'LPR, PVC',
 'PAC, QWAVE',
 'PVC, STD_',
 'PVC, QWAVE',
 'SARRH, VCLVH',
 'QWAVE, STACH',
 'ABQRS, NST_',
 'PVC, VCLVH',
 'NST_, SBRAD',
 'PAC, VCLVH',
 'NDT, NST_',
 'LPR, NT_',
 'AFIB, AFLT',
 'QWAVE, VCLVH',
 'NST_, PVC',
 'STACH, VCLVH',
 'LPR, QWAVE',
 'NDT, SVARR',
 'NDT, PAC',
 'NST_, SARRH',
 'QWAVE, SBRAD',
 'LPR, NDT',
 'LVOLT, STACH',
 'INVT, NDT',
 'ABQRS, LNGQT',
 'NST_, PAC',
 'SVARR, VCLVH',
 'LPR, VCLVH',
 'DIG, STD_',
 'AFIB, NT_',
 'DIG, NST_',
 'PAC, STACH',
 'PVC, SVARR',
 'LVOLT, SBRAD',
 'NST_, NT_',
 'NT_, PVC',
 'INV

In [88]:
#
# Cell 28: TWO LABEL - filters from df "Y_label_len_two_df" all rows with codes in column "codes" 
# that are contained in the list codeslistconvert column "codes" to list
#

Y_label_len_two_short_df= Y_label_len_two_df[Y_label_len_two_df['codes'].isin(codes_two_list)].copy()
Y_label_len_two_short_df

,ecg_id,patient_id,strat_fold,label,codes,len
102,103,13722.0,4,"[ABQRS, STACH]","ABQRS, STACH",2
232,240,9601.0,10,"[ABQRS, SBRAD]","ABQRS, SBRAD",2
253,261,3810.0,8,"[SR, NT_, NDT]","NDT, NT_",2
265,273,340.0,2,"[VCLVH, SR, STD_]","STD_, VCLVH",2
268,276,5087.0,6,"[SR, NT_, NDT]","NDT, NT_",2
...,...,...,...,...,...,...
21745,21784,12488.0,1,"[ABQRS, SR, NDT]","ABQRS, NDT",2
21755,21794,12543.0,1,"[AFIB, NDT]","AFIB, NDT",2
21757,21796,17746.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2
21787,21826,9178.0,10,"[ABQRS, SARRH]","ABQRS, SARRH",2


In [90]:
#
# Cell 29: 
#

Y_label_len_two_short_df.loc[:, 'codes_2'] = 'binary_label'
Y_label_len_two_short_df

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
102,103,13722.0,4,"[ABQRS, STACH]","ABQRS, STACH",2,binary_label
232,240,9601.0,10,"[ABQRS, SBRAD]","ABQRS, SBRAD",2,binary_label
253,261,3810.0,8,"[SR, NT_, NDT]","NDT, NT_",2,binary_label
265,273,340.0,2,"[VCLVH, SR, STD_]","STD_, VCLVH",2,binary_label
268,276,5087.0,6,"[SR, NT_, NDT]","NDT, NT_",2,binary_label
...,...,...,...,...,...,...,...
21745,21784,12488.0,1,"[ABQRS, SR, NDT]","ABQRS, NDT",2,binary_label
21755,21794,12543.0,1,"[AFIB, NDT]","AFIB, NDT",2,binary_label
21757,21796,17746.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2,binary_label
21787,21826,9178.0,10,"[ABQRS, SARRH]","ABQRS, SARRH",2,binary_label


In [92]:
#
# Cell 30: TWO LABEL - filters from df "Y_label_len_two_df" all rows with codes in column "codes" 
# that are contained in the list codeslistconvert column "codes" to list
#

Y_label_len_two_remaining_df= Y_label_len_two_df[Y_label_len_two_df['codes'].isin(codes_two_list_remaining)]
Y_label_len_two_remaining_df

,ecg_id,patient_id,strat_fold,label,codes,len
19,20,13619.0,9,"[ABQRS, AFLT]","ABQRS, AFLT",2
25,26,13619.0,9,"[DIG, NST_, SR]","DIG, NST_",2
27,28,13619.0,9,"[DIG, NST_, SR]","DIG, NST_",2
162,170,15691.0,9,"[PVC, NST_, SR]","NST_, PVC",2
255,263,1595.0,5,"[STACH, NST_]","NST_, STACH",2
...,...,...,...,...,...,...
21731,21770,16077.0,4,"[SARRH, NDT]","NDT, SARRH",2
21739,21778,17965.0,7,"[PVC, STACH]","PVC, STACH",2
21773,21812,20789.0,10,"[ABQRS, SVARR]","ABQRS, SVARR",2
21779,21818,19204.0,10,"[ABQRS, VCLVH, SR]","ABQRS, VCLVH",2


Cell ... to cell ...: combine "Y_label_len_two_short_df" and "Y_label_len_two_short_df"
and substract items from the ... to get the remianing EGS, which are not contained in the two classes

In [94]:
#
# Cell 31: ALL OTHER - combine Y_label_len_one_short_df and Y_label_len_two_short_df
#

Y_one_two_remaining_combined = pd.concat([Y_label_len_one_short_df,
                                Y_label_len_two_short_df, 
                                Y_label_len_one_remaining_df, 
                                Y_label_len_two_remaining_df,
                                Y_SR], verify_integrity=True)
Y_one_two_remaining_combined



,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
0,1,15709.0,3,"[SR, LVOLT]",LVOLT,1,single_label
1,2,13243.0,2,[SBRAD],SBRAD,1,single_label
7,8,11275.0,9,"[ABQRS, SR]",ABQRS,1,single_label
10,11,11243.0,5,[SARRH],SARRH,1,single_label
11,12,11031.0,8,[SBRAD],SBRAD,1,single_label
...,...,...,...,...,...,...,...
21792,21831,11905.0,9,[SR],SR,1,SR
21793,21832,7954.0,7,[SR],SR,1,SR
21796,21835,19311.0,2,[SR],SR,1,SR
21797,21836,8873.0,8,[SR],SR,1,SR


In [96]:
#
# Cell 32: ALL OTHER - substract Y_one_two_combined from Y_copy
#

Y_all_other = Y_copy[~Y_copy.index.isin(Y_one_two_remaining_combined.index)].copy()
Y_all_other


,ecg_id,patient_id,strat_fold,label,codes,len
16,17,13619.0,9,"[ABQRS, AFIB, AFLT]","[ABQRS, AFIB, AFLT]",3
38,39,13619.0,9,"[DIG, NST_, SR, LNGQT, ABQRS]","[ABQRS, DIG, LNGQT, NST_]",4
249,257,6078.0,3,"[STD_, SR, PAC, LOWT]","[LOWT, PAC, STD_]",3
258,266,641.0,9,"[PVC, INVT, SR, QWAVE]","[INVT, PVC, QWAVE]",3
261,269,2282.0,8,"[SR, LPR, PAC, QWAVE]","[LPR, PAC, QWAVE]",3
...,...,...,...,...,...,...
21668,21707,17286.0,7,"[PVC, SR, ABQRS, LNGQT]","[ABQRS, LNGQT, PVC]",3
21703,21742,13857.0,8,"[PVC, AFIB, ABQRS]","[ABQRS, AFIB, PVC]",3
21724,21763,13857.0,8,"[ABQRS, VCLVH, AFIB]","[ABQRS, AFIB, VCLVH]",3
21727,21766,13857.0,8,"[PVC, ABQRS, STACH]","[ABQRS, PVC, STACH]",3


In [98]:
#
# Cell 33: 
#

Y_all_other.loc[:, 'codes_2'] = 'all_other'
Y_all_other

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
16,17,13619.0,9,"[ABQRS, AFIB, AFLT]","[ABQRS, AFIB, AFLT]",3,all_other
38,39,13619.0,9,"[DIG, NST_, SR, LNGQT, ABQRS]","[ABQRS, DIG, LNGQT, NST_]",4,all_other
249,257,6078.0,3,"[STD_, SR, PAC, LOWT]","[LOWT, PAC, STD_]",3,all_other
258,266,641.0,9,"[PVC, INVT, SR, QWAVE]","[INVT, PVC, QWAVE]",3,all_other
261,269,2282.0,8,"[SR, LPR, PAC, QWAVE]","[LPR, PAC, QWAVE]",3,all_other
...,...,...,...,...,...,...,...
21668,21707,17286.0,7,"[PVC, SR, ABQRS, LNGQT]","[ABQRS, LNGQT, PVC]",3,all_other
21703,21742,13857.0,8,"[PVC, AFIB, ABQRS]","[ABQRS, AFIB, PVC]",3,all_other
21724,21763,13857.0,8,"[ABQRS, VCLVH, AFIB]","[ABQRS, AFIB, VCLVH]",3,all_other
21727,21766,13857.0,8,"[PVC, ABQRS, STACH]","[ABQRS, PVC, STACH]",3,all_other


In [100]:
# Cell 34: 

all_classes_dfs ={'SR':Y_SR, 'single':Y_label_len_one_short_df,'binary': Y_label_len_two_short_df,'all_other': Y_all_other} 
all_classes_dfs['binary']

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
102,103,13722.0,4,"[ABQRS, STACH]","ABQRS, STACH",2,binary_label
232,240,9601.0,10,"[ABQRS, SBRAD]","ABQRS, SBRAD",2,binary_label
253,261,3810.0,8,"[SR, NT_, NDT]","NDT, NT_",2,binary_label
265,273,340.0,2,"[VCLVH, SR, STD_]","STD_, VCLVH",2,binary_label
268,276,5087.0,6,"[SR, NT_, NDT]","NDT, NT_",2,binary_label
...,...,...,...,...,...,...,...
21745,21784,12488.0,1,"[ABQRS, SR, NDT]","ABQRS, NDT",2,binary_label
21755,21794,12543.0,1,"[AFIB, NDT]","AFIB, NDT",2,binary_label
21757,21796,17746.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2,binary_label
21787,21826,9178.0,10,"[ABQRS, SARRH]","ABQRS, SARRH",2,binary_label


In [102]:
#
# Cell 35: Selects from the dataframe in individual_dfs all the instances
# with strat_fold == 10 an put them into separate dataframe
#

test_dfs = {name: df[df['strat_fold'] == 10].copy() for name, df in all_classes_dfs.items()}
test_dfs['binary']

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
232,240,9601.0,10,"[ABQRS, SBRAD]","ABQRS, SBRAD",2,binary_label
444,452,13110.0,10,"[PVC, AFIB]","AFIB, PVC",2,binary_label
486,499,2678.0,10,"[SR, NT_, NDT]","NDT, NT_",2,binary_label
568,581,583.0,10,"[AFIB, NST_]","AFIB, NST_",2,binary_label
955,968,4944.0,10,"[INVT, SR, STD_]","INVT, STD_",2,binary_label
...,...,...,...,...,...,...,...
20919,20958,9871.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2,binary_label
21046,21085,19420.0,10,"[PVC, AFIB]","AFIB, PVC",2,binary_label
21054,21093,19420.0,10,"[PVC, AFIB]","AFIB, PVC",2,binary_label
21757,21796,17746.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2,binary_label


In [104]:
#
# Cell 36: substracts for all classes the corresponding test instances from the sets.
# Generates the dictionary "residue_dfs" with all the residue df of the individual classes  
#

residue_dfs = {name: df[~df.index.isin(test_dfs[name].index)].copy() for name, df in all_classes_dfs.items()}

residue_dfs['binary']

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
102,103,13722.0,4,"[ABQRS, STACH]","ABQRS, STACH",2,binary_label
253,261,3810.0,8,"[SR, NT_, NDT]","NDT, NT_",2,binary_label
265,273,340.0,2,"[VCLVH, SR, STD_]","STD_, VCLVH",2,binary_label
268,276,5087.0,6,"[SR, NT_, NDT]","NDT, NT_",2,binary_label
274,282,437.0,5,"[PVC, AFIB]","AFIB, PVC",2,binary_label
...,...,...,...,...,...,...,...
21737,21776,12488.0,1,"[ABQRS, STACH]","ABQRS, STACH",2,binary_label
21740,21779,10590.0,5,"[AFIB, NST_]","AFIB, NST_",2,binary_label
21745,21784,12488.0,1,"[ABQRS, SR, NDT]","ABQRS, NDT",2,binary_label
21755,21794,12543.0,1,"[AFIB, NDT]","AFIB, NDT",2,binary_label


In [124]:
#
# Cell 37: splits the df in the dictionary "residue_dfs" in training and validation sets.
#

training_dfs = {
    name: df.sample(frac=0.75, random_state=7).copy()
    for name, df in residue_dfs.items()
}


validation_dfs = {
    name.replace('_training', '_validation'): df.drop(training_dfs[name].index).copy()
    for name, df in residue_dfs.items()
}



display(training_dfs['binary'])

len(validation_dfs['binary'])

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
7344,7365,8304.0,2,"[ABQRS, AFIB]","ABQRS, AFIB",2,binary_label
18844,18883,17404.0,6,"[PVC, SR, ABQRS]","ABQRS, PVC",2,binary_label
7716,7737,19931.0,5,"[ABQRS, STACH]","ABQRS, STACH",2,binary_label
7527,7548,20927.0,8,"[AFIB, NST_]","AFIB, NST_",2,binary_label
19387,19426,20726.0,8,"[AFIB, NST_]","AFIB, NST_",2,binary_label
...,...,...,...,...,...,...,...
7652,7673,21049.0,6,"[ABQRS, AFIB]","ABQRS, AFIB",2,binary_label
4348,4368,16669.0,8,"[AFIB, NST_]","AFIB, NST_",2,binary_label
3232,3247,7550.0,8,"[VCLVH, SR, STD_]","STD_, VCLVH",2,binary_label
7218,7239,16726.0,6,"[AFIB, NDT]","AFIB, NDT",2,binary_label


325

In [126]:
#
# Cell 38: count the number of instances splits the df in the dictionary "residue_dfs" in training and validation sets.
#

dfs_all= {1: test_dfs,
          2: training_dfs,
          3: validation_dfs
         }

summation=sum(df.shape[0] for df in dfs_all.values() for df in df.values())
print('Total sum of all instances in the different sets: ',summation)

Total sum of all instances in the different sets:  19937


In [128]:
#
# Cell 39: combine all test / validation / training sets of all sets (all classes built)
#

train_df = pd.concat(training_dfs.values(), axis=0).sample(frac=1, random_state=7)
val_df   = pd.concat(validation_dfs.values(), axis=0).sample(frac=1, random_state=7)
test_df  = pd.concat(test_dfs.values(), axis=0).sample(frac=1, random_state=7)
train_df

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
13497,13529,18640.0,1,"[PVC, AFIB, ABQRS]","[ABQRS, AFIB, PVC]",3,all_other
14852,14889,7882.0,8,"[SR, NT_, NDT]","NDT, NT_",2,binary_label
20394,20433,18958.0,4,[SARRH],SARRH,1,single_label
7807,7831,2242.0,4,"[SR, NDT]",NDT,1,single_label
21348,21387,12811.0,7,[SR],SR,1,SR
...,...,...,...,...,...,...,...
2980,2995,13993.0,4,[SR],SR,1,SR
8336,8360,15752.0,3,"[ABQRS, SR]",ABQRS,1,single_label
10254,10281,20883.0,3,[SR],SR,1,SR
2344,2357,18869.0,9,[SARRH],SARRH,1,single_label


In [130]:
#
# Cell 40: 
#

Y_final_dfs = {'train_df' : train_df, 'val_df': val_df, 'test_df' : test_df}

## Following cell (41 to 45) are for preparing X data without augmentation 

In [114]:
#
#Cell 41: loads original X values (no augmentation) 
#

from pathlib import Path

X_directory_path = Path("D:/ECG/X_zero")
X = np.load(X_directory_path/'X_float32.npy', mmap_mode='r')

In [132]:
#
# Cell 42: divides X values in training-, validation- and test data.
#

X_train = X[train_df.index.tolist(), :, :]
X_val   = X[val_df.index.tolist(), :, :]
X_test  = X[test_df.index.tolist(), :, :]

In [118]:
#
# Cell 43: creates a folder for data storage.
#

# Specify the directory name
directory_path = Path("D:/ECG/XY_diff_class_1")

# Create the directory
try:
    directory_path.mkdir()
    print(f"Directory '{directory_path}' created successfully.")
except FileExistsError:
    print(f"Directory '{directory_path}' already exists.")
except PermissionError:
    print(f"Permission denied: Unable to create '{directory_path}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Directory 'D:\ECG\XY_diff_class_1' already exists.


In [134]:
#
# Cell 44: stores the X data (numpy arrays) to archive in the created folder    
#

np.savez(directory_path/'X_np_arrays_3.npz', a = X_train, b = X_val, c = X_test)

In [136]:
#
# Cell 45: Saves Y_final_dfs and X_final_dfs
#

import pickle

with open(directory_path/'Y_data_3.pkl', 'wb') as f:  # open a text file
    pickle.dump(Y_final_dfs, f) # serialize the list
    f.close()

#with open(directory_path/'X_data.pkl', 'wb') as f:  # open a text file
#    pickle.dump(X_final_dfs, f) # serialize the list
#    f.close()

## The following cells must be executed if one wants to use the augmented datasets of X 

In [124]:
#
# Cell 43: path to read input data
#

X_directory_path = Path("D:/ECG/X_augmented_1000") # chose the right path beforhand
X = np.load(X_directory_path/'X_augmented.npz.npy', mmap_mode='r')

In [126]:
#
# Cell 44: divides X values in training-, validation- and test data.
#

X_train = X[train_df.index.tolist(), :, :]
X_val   = X[val_df.index.tolist(), :, :]
X_test  = X[test_df.index.tolist(), :, :]

In [128]:
#
# Cell 45: creates a folder for data storage.
#

# Specify the directory name
directory_path = Path("D:/ECG/XY_diff_class_1_aug_1000") # chose the right path beforhand

# Create the directory
try:
    directory_path.mkdir()
    print(f"Directory '{directory_path}' created successfully.")
except FileExistsError:
    print(f"Directory '{directory_path}' already exists.")
except PermissionError:
    print(f"Permission denied: Unable to create '{directory_path}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Directory 'D:\ECG\XY_diff_class_1_aug_1000' created successfully.


In [130]:
#
# Cell 46: stores the X data (numpy arrays) to archive in the created folder    
#

np.savez(directory_path/'X_np_arrays.npz', a = X_train, b = X_val, c = X_test)

In [132]:
#
# Cell 47: Saves Y_final_dfs and X_final_dfs
#

import pickle

with open(directory_path/'Y_data.pkl', 'wb') as f:  # open a text file
    pickle.dump(Y_final_dfs, f) # serialize the list
    f.close()

#with open(directory_path/'X_data.pkl', 'wb') as f:  # open a text file
#    pickle.dump(X_final_dfs, f) # serialize the list
#    f.close()